In [23]:
import os
import re
import json
import sys
import shutil
import subprocess
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
)

from peft import LoraConfig, get_peft_model


In [14]:
# Get current working directory
BASE_DIR = os.getcwd()
print(f"Base directory: {BASE_DIR}")

# Data paths
RAW_TRAIN_PATH = "data/english/clef_2026_checkthat_english_train.json"
VAL_PATH = "data/english/clef2026_gpt4_o_mini_val.json"
TRAIN_JSONL = "output/training_data_for_RM/english_train_with_evidence.jsonl"

# Output directories
TEACHER_MODEL_DIR = "output/teacher_llama_evidence"
TEACHER_PRED_PATH = "output/RM_prediction/teacher_llama_evidence_predictions.json"
TRAIN_TEACHER_PRED_PATH = "output/RM_prediction/teacher_llama_training_predictions.json"
DISTILL_TRAIN_PATH = "output/training_data_for_distillation.jsonl"
TEACHER_RESULT_DIR = "output/results_teacher_llama_evidence"

# Model config
BASE_MODEL = "meta-llama/Llama-3.2-1B"

# Training params
MAX_LENGTH = 256
BATCH_SIZE = 2
EPOCHS = 3
LR = 1e-4
RANDOM_STATE = 42

# Create directories
os.makedirs("output/training_data_for_RM", exist_ok=True)
os.makedirs("output/RM_prediction", exist_ok=True)
os.makedirs(TEACHER_MODEL_DIR, exist_ok=True)
os.makedirs(TEACHER_RESULT_DIR, exist_ok=True)

Base directory: /home/jovyan/AIR_Group_Task


In [15]:
# =========================
# Run evidence-aware preprocessing script
# =========================
subprocess.run(
    [
        sys.executable,
        "experiments/teacher_llama_evidence/reasoning_trace_build_with_evidence.py",
        "--input",
        RAW_TRAIN_PATH,
        "--output",
        TRAIN_JSONL,
    ],
    check=True,
)

train_df = pd.read_json(TRAIN_JSONL, lines=True)
print(f"Loaded {len(train_df)} training samples")
print(f"Columns: {train_df.columns.tolist()}")
print(f"Class distribution:\n{train_df['Class'].value_counts()}")

Wrote 31433 examples to output/training_data_for_RM/english_train_with_evidence.jsonl
Unknown traces included: 0
Loaded 31433 training samples
Columns: ['sample_id', 'query_id', 'input_text', 'Claim', 'Evidence', 'Justification', 'Label', 'Verdict', 'Class']
Class distribution:
Class
0    22775
1     8658
Name: count, dtype: int64


In [16]:
# =========================
# Helper functions
# =========================

def remove_label_pattern(text):
    text = re.sub(
        r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting))",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()
    return text.replace("\n", " ")

def get_evidence(sample):
    possible_keys = [
        "evidences", "evidence", "Evidence", "relevant_evidence",
        "Relevant_evidence", "context", "Context", "gold_evidence", "Gold_evidence"
    ]
    
    for key in possible_keys:
        if key in sample and sample[key]:
            value = sample[key]
            if isinstance(value, list):
                return " ".join(map(str, value))
            if isinstance(value, dict):
                return json.dumps(value, ensure_ascii=False)
            return str(value)
    return ""

def build_teacher_input(claim, evidence, verdict, justification):
    return (
        f"Claim: {claim}\n"
        f"Evidence: {evidence}\n"
        f"Verdict: {verdict}\n"
        f"Justification: {justification}"
    )

def print_trainable_parameters(model):
    trainable_params = 0
    all_params = 0
    for _, param in model.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || "
        f"all params: {all_params} || "
        f"trainable%: {100 * trainable_params / all_params:.2f}"
    )


In [17]:
# =========================
# Dataset class
# =========================

class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["teacher_input_text"].tolist()
        self.labels = dataframe["Class"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item


In [24]:
# =========================
# Teacher verifier model (LLaMA + LoRA)
# =========================

class TeacherVerifier(torch.nn.Module):
    def __init__(
            self,
            model_name,
            num_labels=1,
            hidden_dim=None,
            dropout_value=0.1,
            use_lora=True,
            lora_rank=8,
            lora_alpha=16,
    ):
        super().__init__()
        
        self.model = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.float32,
        )
        
        if use_lora:
            lora_config = LoraConfig(
                r=lora_rank,
                lora_alpha=lora_alpha,
                target_modules=["q_proj", "k_proj", "v_proj"],
                lora_dropout=0.05,
                bias="none",
            )
            self.model = get_peft_model(self.model, lora_config)
        
        hidden_size = self.model.config.hidden_size
        
        if hidden_dim:
            self.classifier = torch.nn.Sequential(
                torch.nn.Linear(hidden_size, hidden_dim),
                torch.nn.ReLU(),
                torch.nn.Dropout(dropout_value),
                torch.nn.Linear(hidden_dim, num_labels),
            )
        else:
            self.classifier = torch.nn.Linear(hidden_size, num_labels)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        
        pooled_output = outputs.last_hidden_state[:, -1, :]
        pooled_output = pooled_output.float()
        logits = self.classifier(pooled_output)
        return logits


In [25]:
# =========================
# Trainer class
# =========================

class TrainerModule:
    def __init__(
            self,
            model,
            train_loader,
            val_loader,
            epochs,
            lr,
            output_dir,
    ):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs = epochs
        self.optimizer = AdamW(self.model.parameters(), lr=lr, eps=1e-8)
        self.loss_fn = torch.nn.BCEWithLogitsLoss()
        
        total_steps = len(train_loader) * epochs
        warmup_steps = int(0.05 * total_steps)
        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            warmup_steps,
            total_steps,
        )
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
    
    def train(self):
        for epoch in range(self.epochs):
            print(f"\nEpoch {epoch + 1}/{self.epochs}")
            self.model.train()
            
            total_loss = 0
            total_acc = 0
            
            for batch in tqdm(self.train_loader):
                self.optimizer.zero_grad()
                
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)
                
                logits = self.model(input_ids, attention_mask)
                loss = self.loss_fn(logits.squeeze(1), labels)
                
                loss.backward()
                self.optimizer.step()
                self.scheduler.step()
                
                total_loss += loss.item()
                
                preds = (torch.sigmoid(logits).squeeze(1) >= 0.5).detach().cpu().numpy()
                total_acc += accuracy_score(
                    labels.detach().cpu().numpy(),
                    preds,
                )
            
            print(f"Train Loss: {total_loss / len(self.train_loader):.4f}")
            print(f"Train Acc: {total_acc / len(self.train_loader):.4f}")
            self.evaluate(epoch)
    
    def evaluate(self, epoch):
        self.model.eval()
        total_loss = 0
        total_acc = 0
        
        with torch.no_grad():
            for batch in self.val_loader:
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)
                
                logits = self.model(input_ids, attention_mask)
                loss = self.loss_fn(logits.squeeze(1), labels)
                
                total_loss += loss.item()
                
                preds = (torch.sigmoid(logits).squeeze(1) >= 0.5).detach().cpu().numpy()
                total_acc += accuracy_score(
                    labels.detach().cpu().numpy(),
                    preds,
                )
        
        print(f"Val Loss: {total_loss / len(self.val_loader):.4f}")
        print(f"Val Acc: {total_acc / len(self.val_loader):.4f}")
        
        tokenizer.save_pretrained(self.output_dir)
        torch.save(
            self.model.state_dict(),
            os.path.join(self.output_dir, f"teacher_model_epoch_{epoch}.pt"),
        )


In [ ]:
# =========================
# Trainer class
# =========================

class TrainerModule:
    def __init__(
            self,
            model,
            train_loader,
            val_loader,
            epochs,
            lr,
            output_dir,
    ):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs = epochs
        self.optimizer = AdamW(self.model.parameters(), lr=lr, eps=1e-8)
        self.loss_fn = torch.nn.BCEWithLogitsLoss()
        
        total_steps = len(train_loader) * epochs
        warmup_steps = int(0.05 * total_steps)
        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            warmup_steps,
            total_steps,
        )
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
    
    def train(self):
        for epoch in range(self.epochs):
            print(f"\nEpoch {epoch + 1}/{self.epochs}")
            self.model.train()
            
            total_loss = 0
            total_acc = 0
            
            for batch in tqdm(self.train_loader):
                self.optimizer.zero_grad()
                
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)
                
                logits = self.model(input_ids, attention_mask)
                loss = self.loss_fn(logits.squeeze(1), labels)
                
                loss.backward()
                self.optimizer.step()
                self.scheduler.step()
                
                total_loss += loss.item()
                
                preds = (torch.sigmoid(logits).squeeze(1) >= 0.5).detach().cpu().numpy()
                total_acc += accuracy_score(
                    labels.detach().cpu().numpy(),
                    preds,
                )
            
            print(f"Train Loss: {total_loss / len(self.train_loader):.4f}")
            print(f"Train Acc: {total_acc / len(self.train_loader):.4f}")
            self.evaluate(epoch)
    
    def evaluate(self, epoch):
        self.model.eval()
        total_loss = 0
        total_acc = 0
        
        with torch.no_grad():
            for batch in self.val_loader:
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)
                
                logits = self.model(input_ids, attention_mask)
                loss = self.loss_fn(logits.squeeze(1), labels)
                
                total_loss += loss.item()
                
                preds = (torch.sigmoid(logits).squeeze(1) >= 0.5).detach().cpu().numpy()
                total_acc += accuracy_score(
                    labels.detach().cpu().numpy(),
                    preds,
                )
        
        print(f"Val Loss: {total_loss / len(self.val_loader):.4f}")
        print(f"Val Acc: {total_acc / len(self.val_loader):.4f}")
        
        tokenizer.save_pretrained(self.output_dir)
        torch.save(
            self.model.state_dict(),
            os.path.join(self.output_dir, f"teacher_model_epoch_{epoch}.pt"),
        )


In [26]:
# =========================
# Train teacher verifier
# =========================

# Prepare data
train_df["teacher_input_text"] = train_df["input_text"]

train_split, dev_split = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["Class"],
    random_state=RANDOM_STATE,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_dataset = TextDataset(train_split, tokenizer, MAX_LENGTH)
dev_dataset = TextDataset(dev_split, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE)

# Create model
teacher_model = TeacherVerifier(
    model_name=BASE_MODEL,
    use_lora=True,
    lora_rank=8,
    lora_alpha=16,
)

print_trainable_parameters(teacher_model)

# Train
trainer = TrainerModule(
    model=teacher_model,
    train_loader=train_loader,
    val_loader=dev_loader,
    epochs=EPOCHS,
    lr=LR,
    output_dir=TEACHER_MODEL_DIR,
)

trainer.train()

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

trainable params: 1181697 || all params: 1236996097 || trainable%: 0.10

Epoch 1/3


100%|██████████| 12573/12573 [40:29<00:00,  5.18it/s]


Train Loss: 0.5076
Train Acc: 0.7780
Val Loss: 0.4362
Val Acc: 0.8249

Epoch 2/3


100%|██████████| 12573/12573 [40:28<00:00,  5.18it/s]


Train Loss: 0.3676
Train Acc: 0.8419
Val Loss: 0.3629
Val Acc: 0.8462

Epoch 3/3


100%|██████████| 12573/12573 [40:27<00:00,  5.18it/s]


Train Loss: 0.2387
Train Acc: 0.8933
Val Loss: 0.3144
Val Acc: 0.8678


In [28]:
# =========================
# Teacher evaluator class
# =========================

class TeacherEvaluator:
    def __init__(
            self,
            model_path,
            tokenizer_path,
            base_model,
            device="cuda",
    ):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        self.model = TeacherVerifier(
            model_name=base_model,
            use_lora=True,
            lora_rank=8,
            lora_alpha=16,
        )
        
        self.model.load_state_dict(
            torch.load(model_path, map_location=self.device)
        )
        self.model.to(self.device)
        self.model.eval()
    
    def encode_input(self, claim, evidence, verdict, justification, max_length=MAX_LENGTH):
        text = build_teacher_input(
            claim=claim,
            evidence=evidence,
            verdict=verdict,
            justification=justification,
        )
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        return (
            encoding["input_ids"].to(self.device),
            encoding["attention_mask"].to(self.device),
        )
    
    def score(self, claim, evidence, verdict, justification):
        input_ids, attention_mask = self.encode_input(
            claim=claim,
            evidence=evidence,
            verdict=verdict,
            justification=justification,
        )
        with torch.no_grad():
            score = self.model(input_ids, attention_mask).item()
        return float(score)

In [29]:
# =========================
# Generate teacher predictions on TRAINING data
# =========================

BEST_EPOCH = EPOCHS - 1
TEACHER_MODEL_PATH = os.path.join(TEACHER_MODEL_DIR, f"teacher_model_epoch_{BEST_EPOCH}.pt")

print(f"Loading teacher model from: {TEACHER_MODEL_PATH}")

# TRAINING data 
print("Loading TRAINING data...")
train_data = []
with open(TRAIN_JSONL, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            train_data.append(json.loads(line))

print(f"Loaded {len(train_data)} training samples")

# Group by query_id
from collections import defaultdict
queries = defaultdict(list)
for sample in train_data:
    queries[sample['query_id']].append(sample)

print(f"Grouped into {len(queries)} unique queries")

teacher_evaluator = TeacherEvaluator(
    model_path=TEACHER_MODEL_PATH,
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
)

predictions = []

for query_id, samples in tqdm(queries.items(), desc="Generating teacher predictions on training data"):
    claim = samples[0].get("Claim", "")
    evidence = samples[0].get("Evidence", "")
    
    verdict_list = []
    verifier_score_list = []
    justification_list = []
    
    for sample in samples:
        justification = sample.get("Justification", "")
        verdict = sample.get("Verdict", "").lower()
        
        score = teacher_evaluator.score(
            claim=claim,
            evidence=evidence,
            verdict=verdict,
            justification=justification,
        )
        
        verdict_list.append(verdict)
        justification_list.append(justification)
        verifier_score_list.append(score)
    
    best_idx = int(np.argmax(np.array(verifier_score_list)))
    best_verdict = verdict_list[best_idx]
    
    # Get label 
    label = "true" if samples[0].get("Class", 0) == 1 else "false"
    
    predictions.append({
        "query_id": query_id,
        "Claim": claim,
        "Evidence": evidence,
        "Label": label,
        "Verdict_BoN": best_verdict,
        "BoN_Verdict_list": verdict_list,
        "Reasoning_traces": justification_list,
        "score_list": verifier_score_list,
    })


with open(TEACHER_PRED_PATH, "w", encoding="utf-8") as fp:
    json.dump(predictions, fp, indent=4, ensure_ascii=False)

print(f"   File: {TEACHER_PRED_PATH}")
print(f"   Number of predictions: {len(predictions)}")



Loading teacher model from: output/teacher_llama_evidence/teacher_model_epoch_2.pt
Loading TRAINING data...
Loaded 31433 training samples
Grouped into 6400 unique queries


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Generating teacher predictions on training data: 100%|██████████| 6400/6400 [27:43<00:00,  3.85it/s]


   File: output/RM_prediction/teacher_llama_evidence_predictions.json
   Number of predictions: 6400


In [30]:
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

class Config:
    STUDENT_MODEL_NAME = "distilbert-base-uncased"
    TEACHER_PRED_PATH = "output/RM_prediction/teacher_llama_evidence_predictions.json"
    TRAIN_JSONL = "output/training_data_for_RM/english_train_with_evidence.jsonl"
    OUTPUT_DIR = "output/distilled_student_bert"
    MAX_LENGTH = 256
    BATCH_SIZE = 32
    EPOCHS = 3
    LR = 2e-5
    WEIGHT_DECAY = 0.01
    TEMPERATURE = 3.0
    ALPHA_DISTILL = 0.7
    ALPHA_CE = 0.3
    RANDOM_STATE = 42

os.makedirs(Config.OUTPUT_DIR, exist_ok=True)

In [32]:
# ========================================
# DATASET CLASS
# ========================================

class DistillationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length, teacher_logits=None):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.teacher_logits = teacher_logits
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }
        
        if self.teacher_logits is not None:
            item["teacher_logit"] = torch.tensor(self.teacher_logits[idx], dtype=torch.float)
        
        return item

In [33]:
# ========================================
# DATA LOADING
# ========================================

def load_data():
    """Load and prepare training data with teacher logits"""
    train_df = pd.read_json(Config.TRAIN_JSONL, lines=True)
    
    # Create input texts
    if "teacher_input_text" in train_df.columns:
        texts = train_df["teacher_input_text"].tolist()
    elif "input_text" in train_df.columns:
        texts = train_df["input_text"].tolist()
    else:
        texts = train_df.apply(
            lambda row: f"Claim: {row['Claim']}\nEvidence: {row['Evidence']}\nVerdict: {row['Verdict']}\nJustification: {row['Justification']}",
            axis=1
        ).tolist()
    
    # Load teacher predictions
    with open(Config.TEACHER_PRED_PATH, 'r') as f:
        teacher_predictions = json.load(f)
    
    # Extract teacher logits (raw values)
    teacher_logit_map = {}
    for pred in teacher_predictions:
        query_id = pred.get("query_id")
        if query_id is not None and "score_list" in pred and pred["score_list"]:
            teacher_logit_map[query_id] = max(pred["score_list"])
    
    # Align teacher logits with training data
    teacher_logits = []
    for idx, row in train_df.iterrows():
        query_id = row.get("query_id")
        if query_id is not None and query_id in teacher_logit_map:
            teacher_logits.append(teacher_logit_map[query_id])
        else:
            teacher_logits.append(0.0)
    
    # Split data
    train_texts, val_texts, train_labels, val_labels, train_logits, val_logits = train_test_split(
        texts, train_df["Class"].tolist(), teacher_logits,
        test_size=0.2, stratify=train_df["Class"], random_state=Config.RANDOM_STATE
    )
    
    return train_texts, val_texts, train_labels, val_labels, train_logits, val_logits

def get_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(Config.STUDENT_MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token if hasattr(tokenizer, 'eos_token') else '[PAD]'
    return tokenizer


In [34]:
# ========================================
# RESPONSE-BASED DISTILLATION
# ========================================

class ResponseBasedStudent(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased", num_labels=1, dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_labels)
        )
        
        total_params = sum(p.numel() for p in self.parameters())
        print(f"\n Response-Based Student: {total_params:,} ({total_params/1e6:.2f}M) params")
    
    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
            pooled = outputs.pooler_output
        else:
            pooled = outputs.last_hidden_state.mean(dim=1)
        logits = self.classifier(pooled)
        return logits

def train_response_based():
    """Train with response-based distillation (logits only)"""
    
    print("\n" + "="*70)
    print("STRATEGY 1: RESPONSE-BASED DISTILLATION")
    print("="*70)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    
    # Load data
    train_texts, val_texts, train_labels, val_labels, train_logits, val_logits = load_data()
    tokenizer = get_tokenizer()
    
    # Create datasets
    train_dataset = DistillationDataset(train_texts, train_labels, tokenizer, Config.MAX_LENGTH, train_logits)
    val_dataset = DistillationDataset(val_texts, val_labels, tokenizer, Config.MAX_LENGTH, None)
    
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE)
    
    # Model
    student = ResponseBasedStudent(model_name=Config.STUDENT_MODEL_NAME).to(device)
    optimizer = AdamW(student.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)
    total_steps = len(train_loader) * Config.EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
    ce_loss = nn.BCEWithLogitsLoss()
    
    best_val_acc = 0
    best_val_f1 = 0
    history = []
    
    for epoch in range(Config.EPOCHS):
        student.train()
        total_loss = 0
        all_preds, all_labels = [], []
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        for batch in pbar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            teacher_logits = batch["teacher_logit"].to(device).unsqueeze(1)
            
            # Forward pass
            student_logits = student(input_ids, attention_mask)
            
            # Response-based distillation loss
            student_2class = torch.cat([-student_logits, student_logits], dim=1)
            teacher_2class = torch.cat([-teacher_logits, teacher_logits], dim=1)
            
            student_soft = F.log_softmax(student_2class / Config.TEMPERATURE, dim=1)
            teacher_soft = F.softmax(teacher_2class / Config.TEMPERATURE, dim=1)
            
            kl_loss = F.kl_div(student_soft, teacher_soft, reduction='batchmean') * (Config.TEMPERATURE ** 2)
            ce = ce_loss(student_logits.squeeze(1), labels)
            loss = Config.ALPHA_DISTILL * kl_loss + Config.ALPHA_CE * ce
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
            total_loss += loss.item()
            preds = (torch.sigmoid(student_logits).squeeze(1) >= 0.5)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            pbar.set_postfix({'loss': f"{loss.item():.4f}", 'kl': f"{kl_loss.item():.4f}", 'ce': f"{ce.item():.4f}"})
        
        train_acc = accuracy_score(all_labels, all_preds)
        
        # Validation
        student.eval()
        val_preds, val_labels_list = [], []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Evaluating"):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)
                logits = student(input_ids, attention_mask)
                preds = (torch.sigmoid(logits).squeeze(1) >= 0.5)
                val_preds.extend(preds.cpu().numpy())
                val_labels_list.extend(labels.cpu().numpy())
        
        val_acc = accuracy_score(val_labels_list, val_preds)
        p, r, f1, _ = precision_recall_fscore_support(val_labels_list, val_preds, average='binary', zero_division=0)
        
        print(f"\n Epoch {epoch+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}, F1={f1:.4f}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_f1 = f1
            torch.save(student.state_dict(), f"{Config.OUTPUT_DIR}/response_based_best.pt")
        
        history.append({'epoch': epoch+1, 'train_acc': train_acc, 'val_acc': val_acc, 'val_f1': f1})
    
    results = {
        'strategy': 'Response-Based',
        'best_accuracy': best_val_acc,
        'best_f1': best_val_f1,
        'history': history
    }
    
    # Save results
    with open(f"{Config.OUTPUT_DIR}/response_based_results.json", 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"\n Best Acc: {best_val_acc:.4f}, Best F1: {best_val_f1:.4f}")
    return results


In [35]:
# ========================================
# RUN RESPONSE-BASED ONLY
# ========================================

# Run only response-based distillation
response_results = train_response_based()

print(f"\nResponse-Based Results:")
print(f"  Best Accuracy: {response_results['best_accuracy']:.4f}")
print(f"  Best F1: {response_results['best_f1']:.4f}")


STRATEGY 1: RESPONSE-BASED DISTILLATION
Device: cuda


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



 Response-Based Student: 66,658,561 (66.66M) params


Evaluating: 100%|██████████| 197/197 [00:21<00:00,  9.18it/s]



 Epoch 1: Train Acc=0.5874, Val Acc=0.6599, F1=0.4796


Evaluating: 100%|██████████| 197/197 [00:21<00:00,  9.12it/s]



 Epoch 2: Train Acc=0.6608, Val Acc=0.6421, F1=0.4912


Evaluating: 100%|██████████| 197/197 [00:21<00:00,  9.18it/s]



 Epoch 3: Train Acc=0.6675, Val Acc=0.6440, F1=0.4989

 Best Acc: 0.6599, Best F1: 0.4796

Response-Based Results:
  Best Accuracy: 0.6599
  Best F1: 0.4796


In [36]:
# ========================================
# FEATURE-BASED DISTILLATION
# ========================================

class FeatureBasedStudent(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased", num_labels=1, dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        
        # Feature projector
        self.feature_projector = nn.Linear(hidden_size, hidden_size)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_labels)
        )
        
        total_params = sum(p.numel() for p in self.parameters())
        print(f"\n Feature-Based Student: {total_params:,} ({total_params/1e6:.2f}M) params")
    
    def forward(self, input_ids, attention_mask, return_features=False):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        
        if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
            pooled = outputs.pooler_output
        else:
            pooled = outputs.last_hidden_state.mean(dim=1)
        
        features = self.feature_projector(pooled)
        logits = self.classifier(features)
        
        if return_features:
            return logits, features
        return logits

class TeacherFeatureBasedStudent(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased"):
        super().__init__()
        self.teacher = AutoModel.from_pretrained(model_name)
        hidden_size = self.teacher.config.hidden_size
        self.head = nn.Linear(hidden_size, hidden_size)
        
        # Freeze teacher backbone
        for param in self.teacher.parameters():
            param.requires_grad = False
    
    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.teacher(input_ids=input_ids, attention_mask=attention_mask)
            if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
                pooled = outputs.pooler_output
            else:
                pooled = outputs.last_hidden_state.mean(dim=1)
        
        teacher_features = self.head(pooled)
        return teacher_features

def train_feature_based():
    """Train with feature-based distillation"""
    
    print("\n" + "="*70)
    print("STRATEGY 2: FEATURE-BASED DISTILLATION")
    print("="*70)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    
    # Load data
    train_texts, val_texts, train_labels, val_labels, train_logits, val_logits = load_data()
    tokenizer = get_tokenizer()
    
    # Create datasets
    train_dataset = DistillationDataset(train_texts, train_labels, tokenizer, Config.MAX_LENGTH, train_logits)
    val_dataset = DistillationDataset(val_texts, val_labels, tokenizer, Config.MAX_LENGTH, None)
    
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE)
    
    # Models
    student = FeatureBasedStudent(model_name=Config.STUDENT_MODEL_NAME).to(device)
    teacher_featureBased = TeacherFeatureBasedStudent(Config.STUDENT_MODEL_NAME).to(device)
    
    optimizer = AdamW(student.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)
    total_steps = len(train_loader) * Config.EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
    
    ce_loss = nn.BCEWithLogitsLoss()
    mse_loss = nn.MSELoss()
    
    best_val_acc = 0
    best_val_f1 = 0
    history = []
    
    for epoch in range(Config.EPOCHS):
        student.train()
        teacher_featureBased.eval()
        total_loss = 0
        all_preds, all_labels_list = [], []
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        for batch in pbar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            teacher_logits = batch["teacher_logit"].to(device).unsqueeze(1)
            
            # Forward passes
            student_logits, student_features = student(input_ids, attention_mask, return_features=True)
            teacher_features = teacher_featureBased(input_ids, attention_mask)
            
            # Response-based loss 
            student_2class = torch.cat([-student_logits, student_logits], dim=1)
            teacher_2class = torch.cat([-teacher_logits, teacher_logits], dim=1)
            student_soft = F.log_softmax(student_2class / Config.TEMPERATURE, dim=1)
            teacher_soft = F.softmax(teacher_2class / Config.TEMPERATURE, dim=1)
            kl_loss = F.kl_div(student_soft, teacher_soft, reduction='batchmean') * (Config.TEMPERATURE ** 2)
            
            # Feature-based loss 
            feature_loss = mse_loss(student_features, teacher_features)
            
            # Hard label loss
            ce = ce_loss(student_logits.squeeze(1), labels)
            
            # Balanced combination 
            loss = 0.3 * kl_loss + 0.2 * feature_loss + 0.5 * ce
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
            total_loss += loss.item()
            preds = (torch.sigmoid(student_logits).squeeze(1) >= 0.5)
            all_preds.extend(preds.cpu().numpy())
            all_labels_list.extend(labels.cpu().numpy())
            
            pbar.set_postfix({
                'loss': f"{loss.item():.4f}",
                'kl': f"{kl_loss.item():.4f}",
                'feat': f"{feature_loss.item():.4f}",
                'ce': f"{ce.item():.4f}"
            })
        
        train_acc = accuracy_score(all_labels_list, all_preds)
        
        # Validation
        student.eval()
        val_preds, val_labels_list = [], []
        val_probs = []
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Evaluating"):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)
                logits = student(input_ids, attention_mask)
                probs = torch.sigmoid(logits).squeeze(1)
                preds = (probs >= 0.5)
                val_preds.extend(preds.cpu().numpy())
                val_labels_list.extend(labels.cpu().numpy())
                val_probs.extend(probs.cpu().numpy())
        
        val_acc = accuracy_score(val_labels_list, val_preds)
        p, r, f1, _ = precision_recall_fscore_support(val_labels_list, val_preds, average='binary', zero_division=0)
        
        print(f"\n Epoch {epoch+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}, F1={f1:.4f}")
        print(f"   Precision={p:.4f}, Recall={r:.4f}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_f1 = f1
            torch.save(student.state_dict(), f"{Config.OUTPUT_DIR}/feature_based_best.pt")
            print(f" Saved best model (acc: {val_acc:.4f}, f1: {f1:.4f})")
        
        history.append({'epoch': epoch+1, 'train_acc': train_acc, 'val_acc': val_acc, 'val_f1': f1})
    
    results = {
        'strategy': 'Feature-Based',
        'best_accuracy': best_val_acc,
        'best_f1': best_val_f1,
        'history': history
    }
    
    with open(f"{Config.OUTPUT_DIR}/feature_based_results.json", 'w') as f:
        json.dump(results, f, indent=2)
    print(f"   Best Accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")
    print(f"   Best F1: {best_val_f1:.4f}")
    
    return results

In [37]:
# ========================================
# RUN FEATURE-BASED ONLY
# ========================================

# Run only feature-based distillation
feature_results = train_feature_based()

print(f"\nFeature-Based Results:")
print(f"  Best Accuracy: {feature_results['best_accuracy']:.4f}")
print(f"  Best F1: {feature_results['best_f1']:.4f}")


STRATEGY 2: FEATURE-BASED DISTILLATION
Device: cuda


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



 Feature-Based Student: 67,249,153 (67.25M) params


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...


 Epoch 1: Train Acc=0.6282, Val Acc=0.6610, F1=0.4776
   Precision=0.4150, Recall=0.5624
 Saved best model (acc: 0.6610, f1: 0.4776)


Evaluating: 100%|██████████| 197/197 [00:21<00:00,  9.19it/s]



 Epoch 2: Train Acc=0.6696, Val Acc=0.6346, F1=0.4906
   Precision=0.3983, Recall=0.6386


Evaluating: 100%|██████████| 197/197 [00:21<00:00,  9.21it/s]



 Epoch 3: Train Acc=0.6786, Val Acc=0.6439, F1=0.4976
   Precision=0.4070, Recall=0.6403
   Best Accuracy: 0.6610 (66.10%)
   Best F1: 0.4776

Feature-Based Results:
  Best Accuracy: 0.6610
  Best F1: 0.4776


In [38]:
# ========================================
# CONTRASTIVE DISTILLATION
# ========================================

class ContrastiveBasedStudent(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased", num_labels=1, dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_labels)
        )
        
        total_params = sum(p.numel() for p in self.parameters())
        print(f"\n Contrastive-Based Student: {total_params:,} ({total_params/1e6:.2f}M) params")
    
    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
            pooled = outputs.pooler_output
        else:
            pooled = outputs.last_hidden_state.mean(dim=1)
        logits = self.classifier(pooled)
        return logits

def train_contrastive():
    """Train with contrastive distillation"""
    
    print("\n" + "="*70)
    print("STRATEGY 3: CONTRASTIVE DISTILLATION")
    print("="*70)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    
    # Load data
    train_texts, val_texts, train_labels, val_labels, train_logits, val_logits = load_data()
    tokenizer = get_tokenizer()
    
    # Create datasets
    train_dataset = DistillationDataset(train_texts, train_labels, tokenizer, Config.MAX_LENGTH, train_logits)
    val_dataset = DistillationDataset(val_texts, val_labels, tokenizer, Config.MAX_LENGTH, None)
    
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE)
    
    # Model
    student = ContrastiveBasedStudent(model_name=Config.STUDENT_MODEL_NAME).to(device)
    optimizer = AdamW(student.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)
    total_steps = len(train_loader) * Config.EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
    
    ce_loss = nn.BCEWithLogitsLoss()
    
    best_val_acc = 0
    best_val_f1 = 0
    history = []
    
    for epoch in range(Config.EPOCHS):
        student.train()
        total_loss = 0
        all_preds, all_labels = [], []
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        for batch in pbar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            teacher_logits = batch["teacher_logit"].to(device).unsqueeze(1)
            
            # Forward pass
            student_logits = student(input_ids, attention_mask)
            
            # Response-based loss
            student_2class = torch.cat([-student_logits, student_logits], dim=1)
            teacher_2class = torch.cat([-teacher_logits, teacher_logits], dim=1)
            student_soft = F.log_softmax(student_2class / Config.TEMPERATURE, dim=1)
            teacher_soft = F.softmax(teacher_2class / Config.TEMPERATURE, dim=1)
            kl_loss = F.kl_div(student_soft, teacher_soft, reduction='batchmean') * (Config.TEMPERATURE ** 2)
            
            # Contrastive loss 
            student_probs = torch.sigmoid(student_logits).squeeze(1)
            teacher_probs = torch.sigmoid(teacher_logits).squeeze(1)
            
            student_sim = torch.matmul(student_probs.unsqueeze(1), student_probs.unsqueeze(0))
            teacher_sim = torch.matmul(teacher_probs.unsqueeze(1), teacher_probs.unsqueeze(0))
            
            label_mask = torch.eq(labels.unsqueeze(1), labels.unsqueeze(0)).float()
            identity_mask = torch.eye(len(labels), device=device)
            label_mask = label_mask - identity_mask
            
            pos_mask = label_mask > 0
            if pos_mask.any():
                pos_loss = F.mse_loss(student_sim[pos_mask], teacher_sim[pos_mask])
            else:
                pos_loss = torch.tensor(0.0, device=device)
            
            neg_mask = label_mask == 0
            if neg_mask.any():
                neg_loss = F.mse_loss(student_sim[neg_mask], teacher_sim[neg_mask])
            else:
                neg_loss = torch.tensor(0.0, device=device)
            
            contrast_loss = pos_loss + 0.5 * neg_loss
            
            # Combined loss
            ce = ce_loss(student_logits.squeeze(1), labels)
            loss = contrast_loss + 0.5 * kl_loss + 0.2 * ce
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
            total_loss += loss.item()
            preds = (torch.sigmoid(student_logits).squeeze(1) >= 0.5)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            pbar.set_postfix({'loss': f"{loss.item():.4f}", 'contrast': f"{contrast_loss.item():.4f}", 
                            'kl': f"{kl_loss.item():.4f}", 'ce': f"{ce.item():.4f}"})
        
        train_acc = accuracy_score(all_labels, all_preds)
        
        # Validation
        student.eval()
        val_preds, val_labels_list = [], []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Evaluating"):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)
                logits = student(input_ids, attention_mask)
                preds = (torch.sigmoid(logits).squeeze(1) >= 0.5)
                val_preds.extend(preds.cpu().numpy())
                val_labels_list.extend(labels.cpu().numpy())
        
        val_acc = accuracy_score(val_labels_list, val_preds)
        p, r, f1, _ = precision_recall_fscore_support(val_labels_list, val_preds, average='binary', zero_division=0)
        
        print(f"\n Epoch {epoch+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}, F1={f1:.4f}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_f1 = f1
            torch.save(student.state_dict(), f"{Config.OUTPUT_DIR}/contrastive_best.pt")
        
        history.append({'epoch': epoch+1, 'train_acc': train_acc, 'val_acc': val_acc, 'val_f1': f1})
    
    results = {
        'strategy': 'Contrastive',
        'best_accuracy': best_val_acc,
        'best_f1': best_val_f1,
        'history': history
    }
    
    with open(f"{Config.OUTPUT_DIR}/contrastive_results.json", 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"\nBest Acc: {best_val_acc:.4f}, Best F1: {best_val_f1:.4f}")
    return results

In [ ]:
# ========================================
# COMPARE DISTILLATION STRATEGIES 
# ========================================

import time
import re
import shutil
import subprocess
from datetime import datetime


TRAIN_JSONL = Config.TRAIN_JSONL
VAL_DATA_PATH = "data/english/data/english/clef2026_gpt4_o_mini_val.json"

DISTILLATION_STRATEGIES = {
    'response_based': {
        'name': 'Response-Based',
        'model_class': ResponseBasedStudent,
        'model_path': f"{Config.OUTPUT_DIR}/response_based_best.pt",
        'results_file': f"{Config.OUTPUT_DIR}/response_based_results.json"
    },
    'feature_based': {
        'name': 'Feature-Based',
        'model_class': FeatureBasedStudent,
        'model_path': f"{Config.OUTPUT_DIR}/feature_based_best.pt",
        'results_file': f"{Config.OUTPUT_DIR}/feature_based_results.json"
    },
    'contrastive': {
        'name': 'Contrastive',
        'model_class': ContrastiveBasedStudent,
        'model_path': f"{Config.OUTPUT_DIR}/contrastive_best.pt",
        'results_file': f"{Config.OUTPUT_DIR}/contrastive_results.json"
    }
}

AGGREGATIONS = [
    ('top1', lambda verdicts, scores: verdicts[np.argmax(scores)]),
]

def get_hardware_name():
    """Get hardware name for reporting"""
    if torch.cuda.is_available():
        return torch.cuda.get_device_name(0)
    else:
        import platform
        return platform.processor() or "CPU"

def get_parameter_counts(model):
    """Get total and trainable parameter counts"""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

def get_file_size_mb(path):
    """Get file size in MB"""
    return os.path.getsize(path) / (1024 * 1024) if os.path.exists(path) else 0

def sync_if_cuda():
    """Synchronize CUDA if available"""
    if torch.cuda.is_available():
        torch.cuda.synchronize()

In [39]:
# ========================================
# RUN CONTRASTIVE ONLY
# ========================================

# Run only contrastive distillation
contrastive_results = train_contrastive()

print(f"\n Contrastive Results:")
print(f"  Best Accuracy: {contrastive_results['best_accuracy']:.4f}")
print(f"  Best F1: {contrastive_results['best_f1']:.4f}")


STRATEGY 3: CONTRASTIVE DISTILLATION
Device: cuda


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



 Contrastive-Based Student: 66,658,561 (66.66M) params


Evaluating: 100%|██████████| 197/197 [00:21<00:00,  9.20it/s]



 Epoch 1: Train Acc=0.5903, Val Acc=0.6348, F1=0.4744


Evaluating: 100%|██████████| 197/197 [00:21<00:00,  9.18it/s]



 Epoch 2: Train Acc=0.6540, Val Acc=0.6496, F1=0.4904


Evaluating: 100%|██████████| 197/197 [00:21<00:00,  9.17it/s]



 Epoch 3: Train Acc=0.6680, Val Acc=0.6450, F1=0.4939

Best Acc: 0.6496, Best F1: 0.4904

 Contrastive Results:
  Best Accuracy: 0.6496
  Best F1: 0.4904


In [48]:
# ========================================
# COMPARE DISTILLATION STRATEGIES 
# ========================================

import time
import re
import shutil
import subprocess
from datetime import datetime


TRAIN_JSONL = Config.TRAIN_JSONL
VAL_DATA_PATH = "data/english/clef2026_gpt4_o_mini_val.json"

DISTILLATION_STRATEGIES = {
    'response_based': {
        'name': 'Response-Based',
        'model_class': ResponseBasedStudent,
        'model_path': f"{Config.OUTPUT_DIR}/response_based_best.pt",
        'results_file': f"{Config.OUTPUT_DIR}/response_based_results.json"
    },
    'feature_based': {
        'name': 'Feature-Based',
        'model_class': FeatureBasedStudent,
        'model_path': f"{Config.OUTPUT_DIR}/feature_based_best.pt",
        'results_file': f"{Config.OUTPUT_DIR}/feature_based_results.json"
    },
    'contrastive': {
        'name': 'Contrastive',
        'model_class': ContrastiveBasedStudent,
        'model_path': f"{Config.OUTPUT_DIR}/contrastive_best.pt",
        'results_file': f"{Config.OUTPUT_DIR}/contrastive_results.json"
    }
}

AGGREGATIONS = [
    ('top1', lambda verdicts, scores: verdicts[np.argmax(scores)]),
]

def get_hardware_name():
    """Get hardware name for reporting"""
    if torch.cuda.is_available():
        return torch.cuda.get_device_name(0)
    else:
        import platform
        return platform.processor() or "CPU"

def get_parameter_counts(model):
    """Get total and trainable parameter counts"""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

def get_file_size_mb(path):
    """Get file size in MB"""
    return os.path.getsize(path) / (1024 * 1024) if os.path.exists(path) else 0

def sync_if_cuda():
    """Synchronize CUDA if available"""
    if torch.cuda.is_available():
        torch.cuda.synchronize()

In [49]:
# ========================================
# SUMMARY
# ========================================

def evaluate_strategy_on_validation(strategy_key, strategy_config):
    
    print(f"\n{'='*80}")
    print(f" EVALUATING: {strategy_config['name']} STRATEGY")
    print(f"{'='*80}")
    
    if not os.path.exists(strategy_config['model_path']):
        print(f" Model not found: {strategy_config['model_path']}")
        return None
    
    # Load validation data
    with open(VAL_DATA_PATH, 'r', encoding='utf-8') as f:
        val_data = json.load(f)
    print(f"Loaded {len(val_data)} validation samples")
    
    # Load model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    
    model = strategy_config['model_class'](model_name=Config.STUDENT_MODEL_NAME)
    model.load_state_dict(torch.load(strategy_config['model_path'], map_location=device))
    model.to(device)
    model.eval()
    
    tokenizer = get_tokenizer()
    
    # Get model size
    total_params, trainable_params = get_parameter_counts(model)
    model_size_mb = get_file_size_mb(strategy_config['model_path'])
    
    print(f"\n Model Size:")
    print(f"   Total parameters: {total_params:,} ({total_params/1e6:.2f}M)")
    print(f"   Trainable parameters: {trainable_params:,}")
    print(f"   Checkpoint size: {model_size_mb:.2f} MB")
    
    
    scored_samples = []
    total_trace_count = 0
    all_latencies = []
    
    sync_if_cuda()
    inference_start = time.perf_counter()
    
    for idx, sample in enumerate(tqdm(val_data, desc=f"Scoring {strategy_config['name']}")):
        claim = sample.get("claim", "")
        evidence = get_evidence(sample)
        
        verdict_list = []
        score_list = []
        justification_list = []
        
        traces = sample.get("Reasoning_traces", [])
        total_trace_count += len(traces)
        
        for trace_idx, trace in enumerate(traces):
            # Extract justification
            justification = remove_label_pattern(trace).split("Label:")[0]
            
            # Extract verdict
            verdict_list_raw = sample.get("Verdict_list", [])
            if trace_idx < len(verdict_list_raw):
                verdict = verdict_list_raw[trace_idx].lower()
            else:
                verdict = "true"
            
            # Build input
            input_text = build_input(claim, evidence, verdict, justification)
            
            # Tokenize
            inputs = tokenizer(
                input_text,
                truncation=True,
                padding="max_length",
                max_length=Config.MAX_LENGTH,
                return_tensors="pt"
            )
            
            # Measure latency 
            if device.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            
            with torch.no_grad():
                logits = model(inputs["input_ids"].to(device), inputs["attention_mask"].to(device))
                score = torch.sigmoid(logits).squeeze().item()
            
            if device.type == "cuda":
                torch.cuda.synchronize()
            latency_ms = (time.perf_counter() - t0) * 1000
            all_latencies.append(latency_ms)
            
            verdict_list.append(verdict)
            justification_list.append(justification)
            score_list.append(score)
        
        scored_samples.append({
            "query_id": sample.get("query_id", idx),
            "Claim": claim,
            "Evidence": evidence,
            "Label": sample.get("label", ""),
            "verdict_list": verdict_list,
            "score_list": score_list,
            "justification_list": justification_list,
        })
    
    sync_if_cuda()
    inference_end = time.perf_counter()
    
    inference_time_sec = inference_end - inference_start
    avg_time_per_claim_ms = (inference_time_sec / len(val_data)) * 1000
    avg_time_per_trace_ms = (inference_time_sec / total_trace_count) * 1000
    avg_latency_ms = np.mean(all_latencies)
    std_latency_ms = np.std(all_latencies)
    
    print(f"\n Inference Statistics:")
    print(f"   Total inference time: {inference_time_sec:.2f} sec")
    print(f"   Avg per claim: {avg_time_per_claim_ms:.2f} ms")
    print(f"   Avg per trace: {avg_time_per_trace_ms:.2f} ms")
    
    
    results_list = []
    
    for agg_name, agg_fn in AGGREGATIONS:
        print(f"\n   Aggregation: {agg_name}")
        
        # Generate predictions
        predictions = []
        for s in scored_samples:
            best_verdict = agg_fn(s["verdict_list"], s["score_list"])
            predictions.append({
                "query_id": s["query_id"],
                "Claim": s["Claim"],
                "Evidence": s["Evidence"],
                "Label": s["Label"],
                "Verdict_BoN": best_verdict,
                "BoN_Verdict_list": s["verdict_list"],
                "Reasoning_traces": s["justification_list"],
                "score_list": s["score_list"],
            })
        
        # Save predictions for this strategy 
        pred_backup_path = f"output/RM_prediction/{strategy_key}_predictions.json"
        with open(pred_backup_path, "w", encoding="utf-8") as fp:
            json.dump(predictions, fp, indent=4, ensure_ascii=False)
        
        
        SCORER_INPUT = "output/RM_prediction/clef_predictions.json"
        shutil.copy(pred_backup_path, SCORER_INPUT)
        
        # Run scorer
        SCORER_PATH = "task2/scorer.py"
        SCORER_RESULT = "output/RM_prediction/result.csv"
        SCORER_IR = "output/RM_prediction/per_sample_ir.csv"
        
        result = subprocess.run(
            [sys.executable, SCORER_PATH],
            capture_output=True,
            text=True
        )
        
        if result.returncode != 0:
            print(f" Scorer failed: {result.stderr}")
            macro_f1 = float('nan')
            recall_at5 = float('nan')
        else:
            # 
            with open(SCORER_RESULT, 'r') as f:
                content = f.read()
            
            # Extract Macro F1
            macro_match = re.search(r'macro avg,([\d.]+),([\d.]+),([\d.]+)', content)
            macro_f1 = float(macro_match.group(3)) if macro_match else float('nan')
            
            # Extract Recall@5
            recall_match = re.search(r'^5,([\d.]+)', content, re.MULTILINE)
            recall_at5 = float(recall_match.group(1)) if recall_match else float('nan')
            
            # Save scorer results with strategy name
            result_path = f"output/RM_prediction/{strategy_key}_result.csv"
            shutil.copy(SCORER_RESULT, result_path)
            print(f"  Saved scorer results to: {result_path}")
            
            ir_path = f"output/RM_prediction/{strategy_key}_per_sample_ir.csv"
            if os.path.exists(SCORER_IR):
                shutil.copy(SCORER_IR, ir_path)
                print(f"  Saved per-sample IR to: {ir_path}")
        
        results_list.append({
            'Strategy': strategy_config['name'],
            'Aggregation': agg_name,
            'Macro_F1': macro_f1,
            'Recall@5': recall_at5,
            'Total_Parameters': total_params,
            'Trainable_Parameters': trainable_params,
            'Model_Size_MB': model_size_mb,
            'Avg_Latency_ms': avg_latency_ms,
            'Inference_Time_sec': inference_time_sec,
            'Num_Validation_Samples': len(val_data),
            'Num_Traces': total_trace_count
        })
    
    return results_list


all_strategy_results = []

for strategy_key, strategy_config in DISTILLATION_STRATEGIES.items():
    results = evaluate_strategy_on_validation(strategy_key, strategy_config)
    if results:
        all_strategy_results.extend(results)



 EVALUATING: Response-Based STRATEGY
Loaded 1600 validation samples
Device: cuda


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



 Response-Based Student: 66,658,561 (66.66M) params

 Model Size:
   Total parameters: 66,658,561 (66.66M)
   Trainable parameters: 66,658,561
   Checkpoint size: 254.32 MB


Scoring Response-Based: 100%|██████████| 1600/1600 [02:06<00:00, 12.61it/s]


...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



 Contrastive-Based Student: 66,658,561 (66.66M) params

 Model Size:
   Total parameters: 66,658,561 (66.66M)
   Trainable parameters: 66,658,561
   Checkpoint size: 254.32 MB


Scoring Contrastive: 100%|██████████| 1600/1600 [02:07<00:00, 12.52it/s]



 Inference Statistics:
   Total inference time: 127.78 sec
   Avg per claim: 79.86 ms
   Avg per trace: 5.32 ms

   Aggregation: top1
  Saved scorer results to: output/RM_prediction/contrastive_result.csv
  Saved per-sample IR to: output/RM_prediction/contrastive_per_sample_ir.csv


In [52]:
# ========================================
# COMPARISON SUMMARY TABLE
# ========================================

def create_comparison_summary(all_results):
    
    df = pd.DataFrame(all_results)
    
   
    display_cols = [
        'Strategy',
        'Macro_F1',
        'Recall@5',
        'Total_Parameters',
        'Model_Size_MB',
        'Avg_Latency_ms'
    ]
    
    display_df = df[display_cols].copy()
    display_df['Total_Parameters_M'] = display_df['Total_Parameters'] / 1e6
    display_df['Model_Size_MB'] = display_df['Model_Size_MB'].round(1)
    display_df['Avg_Latency_ms'] = display_df['Avg_Latency_ms'].round(1)
    display_df['Macro_F1'] = display_df['Macro_F1'].round(4)
    display_df['Recall@5'] = display_df['Recall@5'].round(4)
    
    display_df.columns = ['Strategy', 'Macro F1', 'Recall@5', 'Total Params', 'Parameters (M)', 
                          'Model Size (MB)', 'Latency (ms)']
    
   
    display_df = display_df[['Strategy', 'Macro F1', 'Recall@5', 'Parameters (M)', 
                              'Model Size (MB)', 'Latency (ms)']]
    
    
    best_macro = df.loc[df['Macro_F1'].idxmax()]
    print(f"   Best Macro F1: {best_macro['Strategy']} ({best_macro['Macro_F1']:.4f})")
    
    best_recall = df.loc[df['Recall@5'].idxmax()]
    print(f"   Best Recall@5: {best_recall['Strategy']} ({best_recall['Recall@5']:.4f})")
    
    best_latency = df.loc[df['Avg_Latency_ms'].idxmin()]
    print(f"   Fastest Inference: {best_latency['Strategy']} ({best_latency['Avg_Latency_ms']:.1f} ms)")
    
    
    smallest_model = df.loc[df['Model_Size_MB'].idxmin()]
    print(f"   Smallest Model: {smallest_model['Strategy']} ({smallest_model['Model_Size_MB']:.1f} MB)")
    
    return df

# Create comparison summary
comparison_df = create_comparison_summary(all_strategy_results)

   Best Macro F1: Feature-Based (0.5052)
   Best Recall@5: Feature-Based (0.3066)
   Fastest Inference: Feature-Based (3.9 ms)
   Smallest Model: Contrastive (254.3 MB)


In [53]:
# ========================================
# SAVE COMPARISON RESULTS TO CSV
# ========================================

def save_comparison_results(all_results, comparison_df):
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = Config.OUTPUT_DIR
    
    # Save formatted summary
    if comparison_df is not None:
        summary_path = f"{output_dir}/strategy_comparison_summary.csv"
        comparison_df.to_csv(summary_path, index=False)
        print(f" Summary saved to: {summary_path}") 

# Save results
save_comparison_results(all_strategy_results, comparison_df)

 Summary saved to: output/distilled_student_bert/strategy_comparison_summary.csv


In [44]:
# ========================================
# PREDICTIONS-DISTILLATION (FINAL TEST DATA)
# ========================================

import sys
import subprocess
import shutil
from pathlib import Path

# Paths
REPO_ROOT = Path(".")
MODEL_DIR = Path(Config.OUTPUT_DIR)
PRED_DIR = Path("output/RM_prediction")
os.makedirs(PRED_DIR, exist_ok=True)

MAX_LENGTH = Config.MAX_LENGTH

def build_input(claim, evidence, verdict, justification):
    """Build input text for the verifier"""
    return f"Claim: {claim}\nEvidence: {evidence}\nVerdict: {verdict}\nJustification: {justification}"

def remove_label_pattern(text):
    """Remove label pattern from justification"""
    if "Label:" in text:
        return text.split("Label:")[0]
    return text

def get_evidence(sample):
    """Extract evidence from sample"""
    if "Evidence" in sample:
        return sample["Evidence"]
    elif "evidence" in sample:
        return sample["evidence"]
    else:
        return sample.get("Evidence_cleaned", "")

class DistilledVerifierEvaluator:
    """Evaluator for distilled student verifiers"""
    
    def __init__(self, model_path, strategy, device="cuda"):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.strategy = strategy
        
        # Load tokenizer
        self.tokenizer = get_tokenizer()
        
        # Load strategy
        if strategy == 'response_based':
            self.model = ResponseBasedStudent(model_name=Config.STUDENT_MODEL_NAME)
        elif strategy == 'feature_based':
            self.model = FeatureBasedStudent(model_name=Config.STUDENT_MODEL_NAME)
        elif strategy == 'contrastive':
            self.model = ContrastiveBasedStudent(model_name=Config.STUDENT_MODEL_NAME)
        else:
            raise ValueError(f"Unknown strategy: {strategy}")
        
        # Load weights
        self.model.load_state_dict(torch.load(model_path, map_location=self.device))
        self.model.to(self.device)
        self.model.eval()
        
        print(f" Loaded {strategy} verifier from {model_path}")
        print(f"   Device: {self.device}")
    
    def encode_input(self, claim, evidence, verdict, justification, max_length=MAX_LENGTH):
        """Encode input for the model"""
        text = build_input(
            claim=claim,
            evidence=evidence,
            verdict=verdict,
            justification=justification,
        )
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        
        return (
            encoding["input_ids"].to(self.device),
            encoding["attention_mask"].to(self.device),
        )
    
    def score(self, claim, evidence, verdict, justification):
        """Get verifier score for a single trace"""
        input_ids, attention_mask = self.encode_input(
            claim=claim,
            evidence=evidence,
            verdict=verdict,
            justification=justification,
        )
        
        with torch.no_grad():
            logits = self.model(input_ids, attention_mask)
            score = torch.sigmoid(logits).squeeze().item()
        
        return float(score)


In [45]:
def generate_predictions_for_strategy(strategy, test_data_path):
    """Generate predictions for a specific distillation strategy"""
    
    print(f"\n{'='*60}")
    print(f"Generating predictions for {strategy.upper()} verifier")
    print(f"{'='*60}")
    
    # Load test data
    with open(test_data_path, "r", encoding="utf-8") as f:
        test_data = json.load(f)
    
    print(f"Loaded {len(test_data)} test samples")
    
    # Load model
    model_path = f"{Config.OUTPUT_DIR}/{strategy}_best.pt"
    if not os.path.exists(model_path):
        print(f" Model not found: {model_path}")
        return None
    
    evaluator = DistilledVerifierEvaluator(
        model_path=model_path,
        strategy=strategy,
        device="cuda"
    )
    
    predictions = []
    
    for idx, sample in enumerate(tqdm(test_data, desc=f"Evaluating {strategy}")):
        claim = sample.get("claim", sample.get("Claim", ""))
        evidence = get_evidence(sample)
        
        verdict_list = []
        verifier_score_list = []
        justification_list = []
        
        # Get number of reasoning traces
        if "Reasoning_traces" in sample:
            num_traces = len(sample["Reasoning_traces"])
        elif "reasoning_traces" in sample:
            num_traces = len(sample["reasoning_traces"])
        else:
            num_traces = len(sample.get("Verdict_list", [1]))
        
        for trace_idx in range(num_traces):
            # Extract justification
            if "Reasoning_traces" in sample:
                justification = remove_label_pattern(
                    sample["Reasoning_traces"][trace_idx]
                ).split("Label:")[0]
            elif "reasoning_traces" in sample:
                justification = remove_label_pattern(
                    sample["reasoning_traces"][trace_idx]
                ).split("Label:")[0]
            else:
                justification = f"Trace {trace_idx}"
            
            # Extract verdict
            if "Verdict_list" in sample:
                verdict = sample["Verdict_list"][trace_idx].lower()
            elif "verdict_list" in sample:
                verdict = sample["verdict_list"][trace_idx].lower()
            else:
                verdict = "true" 
            
            # Get verifier score
            score = evaluator.score(
                claim=claim,
                evidence=evidence,
                verdict=verdict,
                justification=justification,
            )
            
            verdict_list.append(verdict)
            justification_list.append(justification)
            verifier_score_list.append(score)
        
        # Select best trace (top1)
        best_idx = int(np.argmax(np.array(verifier_score_list)))
        best_verdict = verdict_list[best_idx]
        
        # Get ground truth label
        label = sample.get("label", sample.get("Label", ""))
        if isinstance(label, str):
            label = label.lower()
        
        predictions.append({
            "query_id": sample.get("query_id", sample.get("id", idx)),
            "Claim": claim,
            #"Evidence": evidence,
            "Label": label,
            "Verdict_BoN": best_verdict,
            "BoN_Verdict_list": verdict_list,
            "Reasoning_traces": justification_list,
            "score_list": verifier_score_list,
        })
    
    
    pred_path = f"{Config.OUTPUT_DIR}/{strategy}_predictions.json"
    with open(pred_path, "w", encoding="utf-8") as fp:
        json.dump(predictions, fp, indent=4, ensure_ascii=False)
    
    print(f" Saved {len(predictions)} predictions to {pred_path}")
    
    return pred_path

# Test data
VAL_DATA_PATH = "data/english/clef_2026_final_english_test.json"  

prediction_paths = {}

for strategy in ['response_based', 'feature_based', 'contrastive']:
    pred_path = generate_predictions_for_strategy(strategy, VAL_DATA_PATH)
    if pred_path:
        prediction_paths[strategy] = pred_path


Generating predictions for RESPONSE_BASED verifier
Loaded 2558 test samples


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



 Response-Based Student: 66,658,561 (66.66M) params
 Loaded response_based verifier from output/distilled_student_bert/response_based_best.pt
   Device: cuda


Evaluating response_based: 100%|██████████| 2558/2558 [04:12<00:00, 10.14it/s]


...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



 Contrastive-Based Student: 66,658,561 (66.66M) params
 Loaded contrastive verifier from output/distilled_student_bert/contrastive_best.pt
   Device: cuda


Evaluating contrastive: 100%|██████████| 2558/2558 [04:12<00:00, 10.13it/s]


 Saved 2558 predictions to output/distilled_student_bert/contrastive_predictions.json


In [ ]:
# ========================================
# TREC FILES FOR SUBMISSION
# ========================================

def generate_trec_files():
    
    # Create runs directory
    RUNS_DIR = "runs"
    os.makedirs(RUNS_DIR, exist_ok=True)
    
    strategies = ['response_based', 'feature_based', 'contrastive']
    strategy_names = ['Response-Based', 'Feature-Based', 'Contrastive']
    
    trec_files = []
    
    for strategy, name in zip(strategies, strategy_names):
        # Path to predictions file
        pred_path = f"output/RM_prediction/{strategy}_predictions.json"
        
        if not os.path.exists(pred_path):
            print(f"\nPredictions not found for {name}: {pred_path}")
            continue
        
        
        # Load predictions
        with open(pred_path, 'r', encoding='utf-8') as f:
            predictions = json.load(f)
        
        print(f"   Loaded {len(predictions)} queries")
        
        trec_filename = f"run_distilbert_{strategy}.txt"
        trec_path = os.path.join(RUNS_DIR, trec_filename)
        
        # Generate TREC format
        with open(trec_path, 'w', encoding='utf-8') as out:
            for sample in predictions:
                query_id = sample["query_id"]
                score_list = sample["score_list"]
                
                # Sort traces by score 
                ranked_traces = sorted(
                    enumerate(score_list),
                    key=lambda x: x[1],
                    reverse=True
                )
                
                for rank, (trace_idx, score) in enumerate(ranked_traces, start=1):
                    trace_id = f"{query_id}_{trace_idx}"
                    run_tag = f"distilbert_{strategy}"
                    out.write(f"{query_id}\tQ0\t{trace_id}\t{rank}\t{score:.6f}\t{run_tag}\n")
        
        # Check file size
        file_size = os.path.getsize(trec_path) / 1024  
        print(f"   TREC file saved: {trec_path}")
        print(f"   File size: {file_size:.1f} KB")
        
        # Count lines
        with open(trec_path, 'r') as f:
            line_count = sum(1 for _ in f)
        print(f"   Lines: {line_count:,}")
        
        trec_files.append({
            'strategy': name,
            'file_path': trec_path,
            'filename': trec_filename,
            'lines': line_count
        })
    
    return trec_files

# Generate TREC files
trec_files = generate_trec_files()